# Train the SLM team for free (Kaggle / Colab GPU, no Modal)

Kaggle: *Settings → Accelerator → GPU T4 x2*, *Internet on* (30 free GPU-hours/week). Colab: *Runtime → T4 GPU*.

Trains the four specialists (Qwen3-0.6B + LoRA) and the manager (Qwen3-1.7B + LoRA) with the plain backprop loop in `slm/train/backprop.py`, then zips the adapters in the Modal Volume layout.

In [ ]:
# The repo is private: add a GitHub token as a secret named GH_TOKEN
# (Kaggle: Add-ons → Secrets; Colab: the key icon → Secrets), read access to NOVA-SLM/nova-hacks is enough.
import os
try:
    from kaggle_secrets import UserSecretsClient; os.environ['GH_TOKEN'] = UserSecretsClient().get_secret('GH_TOKEN')
except ImportError:
    from google.colab import userdata; os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
!git clone -q https://$GH_TOKEN@github.com/NOVA-SLM/nova-hacks.git repo 2>/dev/null || git -C repo pull -q
%cd repo
!pip -q install -e . "peft==0.17.1" "transformers==4.57.1" accelerate

In [ ]:
!python -m slm.train build-data --root out
!python -m slm.train estimate --root out

Smoke test first (≈2 min): 32 examples, 1 epoch, 8 gate rows on one specialist.

In [ ]:
!python -m slm.train backprop --role auth --root out --max-train 32 --epochs 1 --gate-limit 8

Full run: four specialists, then the manager. On a T4 expect roughly 10–20 min per specialist and 15–30 min for the manager.

In [ ]:
!python -m slm.train backprop --role specialists --root out
!python -m slm.train backprop --role manager --root out --micro-batch 1

In [ ]:
!cd out && zip -qr ../slm-adapters.zip adapters manager data/manifest.json && cd .. && ls -lh slm-adapters.zip
# Kaggle: download from the Output tab.  Colab: from google.colab import files; files.download('slm-adapters.zip')

**Use the adapters**: unzip into `out/` locally and run `uv run slm ui` (Local adapters backend, needs torch), or upload them for the vLLM server: `modal volume put slm-vol out/adapters /adapters`.